# Data Cleaning — Match Context

In [22]:
import pandas as pd

df = pd.read_csv('../../data/raw/gold_match_context.csv', sep=',', on_bad_lines='skip')

# Filter to home games only using is_home_match from gold_match
gold_match = pd.read_csv('../../data/raw/gold_match.csv', sep=',', on_bad_lines='skip')
home_ids = gold_match[gold_match['is_home_match'] == True]['match_id']
df = df[df['match_id'].isin(home_ids)].reset_index(drop=True)

print(df.shape)
df.head()

(71, 35)


,match_id,match_date,weather_temp_max_c,weather_temp_min_c,weather_temp_mean_c,weather_precipitation_mm,weather_rain_mm,weather_snowfall_cm,weather_windspeed_max_kmh,weather_sunshine_hours,...,is_weekend,is_midweek,academic_year,academic_week,is_public_holiday,public_holiday_name,is_school_holiday_flanders,school_holiday_name,pct_free_tickets,campaign_motto
0,d256yo3eng04m0fu7b4sl7wno,2022-07-30,25.8,12.9,19.8,0.0,0.0,0.0,10.2,10.3,...,True,False,2021/2022,48,False,NaN,True,Summer Holiday,13.9,NaN
1,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,32.2,17.3,25.4,0.0,0.0,0.0,16.9,11.1,...,True,False,2021/2022,50,False,NaN,True,Summer Holiday,8.3,NaN
2,d65hmi7sq03yzr5he1k7ypus4,2022-08-27,21.0,15.9,18.7,0.0,0.0,0.0,13.7,3.9,...,True,False,2021/2022,52,False,NaN,True,Summer Holiday,14.8,NaN
3,d80mkemezkz16bqh6lbn8tlhw,2022-09-10,18.9,13.6,15.7,8.6,8.6,0.0,23.5,4.1,...,True,False,2022/2023,2,False,NaN,False,NaN,68.7,NaN
4,dak40etbhbqsr1nxyt50qcg0k,2022-10-01,17.8,10.9,14.2,9.8,9.8,0.0,34.5,8.5,...,True,False,2022/2023,5,False,NaN,False,NaN,7.5,NaN


In [23]:
# Fix data types
df['match_date'] = pd.to_datetime(df['match_date'])

bool_cols = ['is_weekend', 'is_midweek', 'is_public_holiday', 'is_school_holiday_flanders', 'has_promotion']
for col in bool_cols:
    df[col] = df[col].map({'true': True, 'false': False}).astype(bool)

In [24]:
# Create binary flags before dropping source columns
df['has_campaign'] = df['campaign_motto'].notna()
df['has_snow'] = df['weather_snowfall_cm'] > 0

# Drop redundant, leaky, and text columns
cols_to_drop = [
    # Original cleaning pass
    'weekday_name',
    'weather_code',
    'weather_precipitation_mm',
    'weather_snowfall_cm',       # replaced by has_snow
    'pct_free_tickets',          # data leakage
    'campaign_motto',            # replaced by has_campaign
    'promotion_names',
    'public_holiday_name',
    'school_holiday_name',
    # Second pass
    'weather_temp_max_c',        # summarised by weather_score + weather_temp_deviation
    'weather_temp_min_c',
    'weather_temp_mean_c',
    'weather_description',       # Dutch text, already captured by weather_score
    'weekday',                   # redundant with is_weekend and is_midweek
    'academic_year',             # season-level signal covered by engineered features
    'promo_valentijn_tickets',   # too granular for 71 matches; has_promotion + promo_tickets_total sufficient
    'promo_1plus1_tickets',
    'promo_bring_a_friend_tickets',
    'promo_free_subscriber_tickets',
    'promo_bondskaarten_tickets',
    'promo_discount_tickets',
]
df = df.drop(columns=cols_to_drop)

print(df.shape)
print(df.dtypes)

(71, 16)
match_id                                 str
match_date                    datetime64[us]
weather_rain_mm                      float64
weather_windspeed_max_kmh            float64
weather_sunshine_hours               float64
weather_temp_deviation               float64
weather_score                        float64
has_promotion                           bool
promo_tickets_total                    int64
is_weekend                              bool
is_midweek                              bool
academic_week                          int64
is_public_holiday                       bool
is_school_holiday_flanders              bool
has_campaign                            bool
has_snow                                bool
dtype: object


In [25]:
# df.to_csv('../../data/cleaned/gold_match_context_cleaned.csv', index=False)
# print('Saved.')